In [1]:


import numpy as np
import pydicom
import cv2
from pydicom.pixel_data_handlers.util import apply_voi_lut

def enhance_bone(img):
    # Passage en 8-bit pour le filtre CLAHE
    img_8bit = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img_8bit)
    return enhanced.astype(np.float32) / 255.0

def preprocess_dicom(path, size=224):
    try:
        dicom = pydicom.dcmread(path)
        img = dicom.pixel_array
        
        # Appliquer le contraste natif du scanner
        if 'VOILUTSequence' in dicom or 'WindowCenter' in dicom:
            img = apply_voi_lut(img, dicom)
        
        # Normalisation Min-Max [0, 1]
        img = img - np.min(img)
        if np.max(img) != 0:
            img = img / np.max(img)
        
        # Amélioration CLAHE pour voir les fractures
        img = enhance_bone(img)
        
        # Redimensionnement standard pour ResNet
        img = cv2.resize(img, (size, size))
        return img
    except Exception as e:
        return np.zeros((size, size), dtype=np.float32)